---------------------------------------------------------Part 3 Face Emotion model----------------------------------------------------
The transformer here is it looks at the face of the person face is been tracked for the whole thing suppose the guy is angry or in some kind of emotion we moniotre the face features for like the whole thing like for the block size and then we try to predict how exaclty the the person face should be now we also have the feature of where the predicted face and the emotion which we have how different are those so our model train on those things as well and also the loss in the emotions adn all of our training face emotions like that we get the personlity of the whole face features as per the emotions then also the addition of vqvae obv shortens our data as per the most faces are and we can get the codebook which are repeating and then use them to train the models

In [ ]:
from Processing_data import Speech_to_mag , VQVAE 
from STT_and_TTS import TTS_and_STT , Head , head_concat , block_in_transformer , STT_transformer_basic_mode , TTS_transformer 

In [ ]:
from typing import dataclass_transform
import torch.nn as nn
import cv2
import mediapipe as mp
from google.colab import files
import cv2
import mediapipe as mp

fsr = FSR()
fsr.make_data(dataset)
fsr.defi()
fsr.defining_part()
fsr.training()
fsr.save()
codebook_face_vqvae_data  = 0

class FSR(nn.Module):
        def __init__(self):
          super().__init__()
          self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
          self.vocab_size = 0
          self.dim = 256
          self.n_layers = 4
          self.num_heads = 4
          self.head_quant = 64
          self.block_size = 128
          self.batch_size = 64
          self.positions = nn.Embedding(self.block_size, self.dim).to(self.device)
          self.x_tokens   = []
          self.y_tokens  = []
          self.z_tokens = []

        def make_data(self, dataset):
            mp_face_mesh = mp.solutions.face_mesh
            with mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True) as face_mesh:
              for emo_token, file_ in enumerate(dataset):
                data = []
                for photo in file_:
                    rgb = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)
                    result = face_mesh.process(rgb)
                    data.append(result)
                vqvae = VQVAE(data, 468, 128)
                vqvae.forward()
                self.vocab_size = vqvae.get_vocab_size()
                codebook_face_vqvae_data =  self.vocab_size
                token_ids  = torch.tensor(list(vqvae.get_tokens()))
                self.x_tokens.append(token_ids[:-1])
                self.y_tokens.append(token_ids[1:])
                self.z_tokens.append(torch.full((token_ids.shape[0] - 1,), emo_token))

        def defi(self) :
              self.Blocks = nn.ModuleList([block_in_transformer() for _ in range(self.n_layers)]).to(self.device)
              self.Encods = nn.Linear(256, self.dim).to(self.device)
              self.face = nn.Linear(self.dim, self.vocab_size).to(self.device)
              self.emotion = nn.Linear(self.dim, 8).to(self.device)

        def custom_data(self, emotion , dataset):
            mp_face_mesh = mp.solutions.face_mesh
            with mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True) as face_mesh:
              for file_ in dataset:
                data = []
                for photo in file_:
                    rgb = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)
                    result = face_mesh.process(rgb)
                    data.append(result)
                vqvae = VQVAE(data, 468, 128)
                vqvae.forward()
                token_ids  = torch.tensor(list(vqvae.get_tokens()))
                self.x_tokens.append(token_ids[:-1])
                self.y_tokens.append(token_ids[1:])
                self.z_tokens.append(torch.full((token_ids.shape[0] - 1,), emotion))

        def defining_part(self):
            self.x_tokens = torch.stack(self.x_tokens)
            self.y_tokens = torch.stack(self.y_tokens)
            self.z_tokens = torch.stack(self.z_tokens)
            self.xs = []
            self.ys  = []
            self.zs = []

            for i in range(0, len(self.x_tokens) - self.block_size):
                self.xs.append(self.x_tokens[i : i + self.block_size])
                self.ys.append(self.y_tokens[i : i + self.block_size])
                self.zs.append(self.z_tokens[i : i + self.block_size])

            self.xs = torch.stack(self.xs)
            self.ys = torch.stack(self.ys)
            self.zs = torch.stack(self.zs)

        def training(self):
            self.parameters = list(self.Encods.parameters()) + list(self.face.parameters()) +list(self.emotion.parameters()) + list(self.positions.parameters()) +list(self.Blocks.parameters())
            self.optimizer = torch.optim.AdamW( self.parameters , lr=1e-4)
            for i in range(10000):
                ix  = torch.randint(0, self.xs.shape[0], (self.batch_size,))
                x_  = self.xs[ix].to(self.device)
                y_  = self.ys[ix].to(self.device)
                z_  = self.zs[ix].to(self.device)
                tok_emb = self.Encods(x_)
                pos_emb = self.positions(torch.arange(self.block_size).to(self.device))
                x = tok_emb + pos_emb

                for block in self.Blocks:
                      x = block(x)

                face_logits     = self.face(x)
                emotions_logits = self.emotion(x)

                face_loss     = torch.nn.functional.cross_entropy(face_logits.view(-1, self.vocab_size), y_.view(-1))
                emotion_face_loss  = torch.nn.functional.cross_entropy(face_logits.view(-1, self.vocab_size),z_.view(-1))
                relation_loss = torch.nn.functional.cross_entropy(emotions_logits.view(-1, 8),y_.view(-1))
                loss = face_loss + emotion_face_loss + relation_loss
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                if i % 500 == 0:
                    print(f'step {i} loss: {loss.item():.4f}')
                if loss.item() < 1.3:
                    print(f'stopping at step {i}')
                    break

        def predict(self, emotion  , video_path ):
            webcam = cv2.VideoCapture(video_path)
            frames = []
            while True:
                ret, frame = webcam.read()
                if not ret:
                    break
                frames.append(frame)
            webcam.release()
            predications = []
            for i in range(0, len(frames),10)
                chunk = frames[i:i+10]
                self.custom_data(emotion , [chunk])
                token_ids = self.x_tokens[-1].unsqueeze(0).to(self.device)
                tok_emb = self.Encods(token_ids)
                pos_emb = self.positions(torch.arange(token_ids.shape[1]).to(self.device))
                x = tok_emb + pos_emb
                for block in self.Blocks:
                    x = block(x)
                predications.append(torch.argmax(self.face(x), dim=-1))
             return predications

       def augmentated_data(self, emotion  , frames ):
            predications = []
            for i in range(0, len(frames),10) :
                temp = []
                chunk = frames[i:i+10]
                self.custom_data(emotion , [chunk])
                token_ids = self.x_tokens[-1].unsqueeze(0).to(self.device)
                tok_emb = self.Encods(token_ids)
                pos_emb = self.positions(torch.arange(token_ids.shape[1]).to(self.device))
                x = tok_emb + pos_emb
                for block in self.Blocks:
                    x = block(x)
                face = self.face(x)
                orginal_pred = self.Encods(torch.argmax(face, dim=-1))
                neighbours = torch.topk(face, k=6, dim=-1)
                neighbours_indices = neighbours.indices
                neighbours_values = neighbours.values
                neighbours_encod = self.Encods(neighbours_indices)
                temp = [(0.9 *orginal_pred + 0.1 * neighbours_encod) , (0.8 *orginal_pred + 0.2 * neighbours_encod) ,(0.7 *orginal_pred + 0.3* neighbours_encod) ]
                neighbours_data.append(temp)
             return neighbours_data

        def save(self):
            torch.save({'Encods': self.Encods.state_dict(),'positions': self.positions.state_dict(),'blocks': self.Blocks.state_dict(),'face': self.face.state_dict(),'emotion': self.emotion.state_dict(), }, 'Emotion_face_transformer.pt')
            files.download('Emotion_face_transformer.pt')